In [1]:
from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
from qonnx.core.modelwrapper import ModelWrapper
import os
from os.path import join
import shutil
import numpy as np
    
build_dir = os.environ["FINN_BUILD_DIR"]
finn_root = os.environ["FINN_ROOT"]
build_dir
# finn_root

'/home/vision/danilowi/serious_mot/build_dirs/brevitas_tests'

In [2]:
!pip install opencv-python
!pip install py-cpuinfo

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [15]:
!pip list
# !pip install brevitas==0.10.2


Package                   Version                               Editable project location
------------------------- ------------------------------------- -----------------------------------------------------------------
anyio                     4.6.2.post1
argon2-cffi               23.1.0
argon2-cffi-bindings      21.2.0
arrow                     1.3.0
asttokens                 2.4.1
async-lru                 2.0.4
attrs                     24.2.0
babel                     2.16.0
backcall                  0.2.0
beautifulsoup4            4.12.3
bitstring                 3.1.7
bleach                    6.2.0
brevitas                  0.10.3.dev90+gd4834bd                 /home/vision/danilowi/serious_mot/finn/deps/brevitas/src
cachetools                5.5.0
certifi                   2024.8.30
cffi                      1.17.1
cfgv                      3.4.0
charset-normalizer        3.4.0
clize                     5.0.1
coloredlogs               15.0.1
comm                      0.2.2
Co

In [7]:
import torch

ckpt = torch.load('quantyolov8_4w4a_comact/weights/best.pt', map_location='cpu')
ckpt_indact = torch.load('quantyolov8_4w4a_indact/weights/best.pt', map_location='cpu')

print(len(ckpt['ema']))
print(len(ckpt_indact['ema']))

ema_dict = ckpt['ema']
ema_dict_indact = ckpt_indact['ema']
# for k, v in ema_dict.items():
#     if k not in ema_dict_indact:
#         print(k)

# for k, v in ema_dict_indact.items():
#     if k not in ema_dict:
#         print(k)
#     ema_dict[k] = ckpt_comact['ema'][k]
#     # if k not in ckpt_comact:
#     #     print(k)

# ckpt_indact['ema'] = ema_dict
# torch.save(ckpt_indact, 'best_v8_comact_clean.pt')
for k, v in ema_dict.items():
    print(k)
    if 'scaling' in k:
        print(v)


420
412
model.0.conv.weight
model.0.bn.weight
model.0.bn.bias
model.0.bn.running_mean
model.0.bn.running_var
model.0.bn.num_batches_tracked
model.0.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5850)
model.1.conv.weight
model.1.bn.weight
model.1.bn.bias
model.1.bn.running_mean
model.1.bn.running_var
model.1.bn.num_batches_tracked
model.1.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5850)
model.2.common_act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5850)
model.2.cv1.conv.weight
model.2.cv1.bn.weight
model.2.cv1.bn.bias
model.2.cv1.bn.running_mean
model.2.cv1.bn.running_var
model.2.cv1.bn.num_batches_tracked
model.2.cv1.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5850)
model.2.cv2.conv.weight
model.2.cv2.bn.weight
model.2.cv2.bn.bias
model.2.cv2.bn.running_mean
model.2.cv2.bn.running_var
model.2.cv2.bn.num_batches_tracked
model.2.cv2.a

In [2]:
from models.yolo import get_model
from models.finn_models import QuantV8Detect, QuantC2f, QuantDetect

# MODEL_PREFIX = "yololit"
# net, _, _ = get_model('yololit.yaml', 'best_yololit.pt', backbone_only=False, load_ema=True)
# test_input_np = np.load(join(finn_root, "notebooks", "experiments", "yolov8", "real_test_input_320x320.npy"))

# MODEL_PREFIX = "quantyolov8_4w4a"
# MODEL_PREFIX = "quantyolov8_4w4a_indact"
MODEL_PREFIX = "quantyolov8_4w4a_comact"

net, _, _ = get_model(MODEL_PREFIX + '/cfg.yaml', MODEL_PREFIX + '/weights/best.pt', backbone_only=False, load_ema=True)
# net, _, _ = get_model(MODEL_PREFIX + '/cfg.yaml', "quantyolov8_4w4a_comact" + '/weights/best.pt', backbone_only=True, load_ema=True)
test_input_np = np.load(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX, "test_input_192x320.npy"))

# net.eval()
net.eval()
for m in net.modules():
    # print(type(m))
    if isinstance(m, QuantC2f):
        m.forward = m.forward_split
    if isinstance(m, QuantV8Detect) or isinstance(m, QuantDetect):
        m.finn_export = True


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/usr/local/lib/python3.10/dist-packages/torch/nn/modules/conv.py:459: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  return F.conv2d(input, weight, bias, self.stride,


saving features[0]:  QuantConv
saving features[1]:  QuantConv
saving features[2]:  QuantC2f
saving features[3]:  QuantConv
saving features[4]:  QuantC2f
saving features[5]:  QuantConv
saving features[6]:  QuantC2f
saving features[7]:  QuantConv
saving features[8]:  QuantC2f
saving features[9]:  QuantSPPF
GET_MODEL: 'ema' loaded
Transferred 420/420 items from quantyolov8_4w4a_comact/weights/best.pt


In [3]:
# QONNX EXPORT

import cv2
import torch
import numpy as np
from brevitas.export import export_qonnx

from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup_model
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import (
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
)
import onnx
from onnx import helper as oh

from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.util.pytorch import ToTensor
from ultralytics.nn.modules import C2f, QuantC2f, QuantDetect



# test input
# test_input_np = (np.random.rand(1, 3, 384, 640) * 255).astype(np.uint8)
# np.save(join(finn_root, "notebooks", "experiments", "yolov8", "test_input_384x640.npy"), test_input_np)

test_input = torch.from_numpy(test_input_np).float() / 255.0

# net = net.eval()
# for m in net.modules():
#     if isinstance(m, C2f) or isinstance(m, QuantC2f):
#         m.forward = m.forward_split
#     if isinstance(m, QuantDetect):
#         m.finn_export = True

# weights
# print(net.state_dict().keys())
# torch.save(net.state_dict(), join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_test_weights.pt"))
# net.load_state_dict(torch.load(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_test_weights.pt")))

print('INPUT:', test_input.shape)
print(test_input)
output_golden = net(test_input)
saved_features = net.saved_features
print('SAVED FEATURES:')
for i, f in enumerate(saved_features):
    gold = np.load('{}/{}_f{}.npy'.format(MODEL_PREFIX, MODEL_PREFIX, i))
    diffs = np.abs(f.detach().cpu().numpy() - gold)
    mask = diffs
    # mask = np.sum(diffs, axis=1)
    # mask = mask / np.max(mask) * 255
    mask = (mask > 0)
    print(i, np.sum(mask) / np.product(mask.shape) * 100, "% wrong", "max error:", np.max(diffs))
    # print(np.argwhere(mask))
    # print('errvalues:', 
    
    # bgr = np.concatenate([np.zeros(mask.shape), np.zeros(mask.shape), mask], axis=0).transpose(1, 2, 0)
    # print(bgr.shape)
    # cv2.imwrite('testbgr.jpg', bgr)
    # print(i, "max error:", np.max(diffs))
    # if i == 0:
    #     print(f.shape)
    #     print(f)
# print(net.saved_features[0].shape)
# print(net.saved_features[0])
# print('OUTPUT:', [x.shape for x in output_golden] if isinstance(output_golden, list) or isinstance(output_golden, tuple) else output_golden.shape)

# print('OUTPUT:')
# for i, out in enumerate(output_golden):
#     gold = np.load('{}/{}_trainout{}.npy'.format(MODEL_PREFIX, MODEL_PREFIX, i))
#     diffs = np.abs(out.detach().cpu().numpy() - gold)
#     mask = diffs
#     # mask = np.sum(diffs, axis=1)
#     # mask = mask / np.max(mask) * 255
#     mask = (mask > 0)
#     print(i, np.sum(mask) / np.product(mask.shape) * 100, "% wrong", "max error:", np.max(diffs))
    
# print(output_golden[0].shape)
# print(output_golden[0])
# print(output_golden)
# for i, out in enumerate(output_golden):
#     np.save(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_output_golden_{}.npy".format(i)), out.detach())

# onnx export
exportmodel_filename = join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_export.onnx")
tidymodel_filename = join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_tidy.onnx")
rawmodel_filename = join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_raw.onnx")
onnx_model = export_qonnx(net, test_input, exportmodel_filename)
model = ModelWrapper(exportmodel_filename)

# onnx resize node workaround
# dummy = oh.make_tensor_value_info("dummy", onnx.TensorProto.FLOAT, [1])
# for n in model.graph.node:
#     if n.op_type == "Resize":
#         n.input[1] = dummy.name
# model.graph.value_info.append(dummy)
model = model.transform(InferShapes())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())

model.save(rawmodel_filename)
model = cleanup_model(model)
model = model.transform(ConvertQONNXtoFINN())

# add preprocessing
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
chkpt_preproc_name = join(build_dir, "preproc.onnx")
export_qonnx(ToTensor(), torch.randn(ishape), chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = cleanup_model(pre_model)
pre_model = pre_model.transform(ConvertQONNXtoFINN())
model = model.transform(MergeONNXModels(pre_model))
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# final cleanup
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

# onnx resize node workaround
# dummy = oh.make_tensor_value_info("dummy", onnx.TensorProto.FLOAT, [1])
# for n in model.graph.node:
#     if n.op_type == "Resize":
#         n.input[1] = dummy.name
# model.graph.value_info.append(dummy)

model.save(tidymodel_filename)
# model.save(join(finn_root, "notebooks", "experiments", "yolov8", "quantyolov8_qonnx.onnx"))

INPUT: torch.Size([1, 3, 192, 320])
tensor([[[[0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          ...,
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471]],

         [[0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          ...,
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471]],

         [[0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
          [0.4471, 0.4471, 0.4471,  ..., 0.4471, 0.4471, 0.4471],
      

/home/vision/danilowi/serious_mot/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [4]:
import cv2
import torch
import numpy as np
from brevitas.export import export_qonnx

from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup_model
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import (
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
)
import onnx
from onnx import helper as oh

from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.util.pytorch import ToTensor
from ultralytics.nn.modules import C2f, QuantC2f, QuantDetect


model = ModelWrapper('export.onnx')

# onnx resize node workaround
dummy = oh.make_tensor_value_info("dummy", onnx.TensorProto.FLOAT, [1])
for n in model.graph.node:
    if n.op_type == "Resize":
        n.input[1] = dummy.name
model.graph.value_info.append(dummy)

model.save(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_raw.onnx"))
model = cleanup_model(model)
model = model.transform(ConvertQONNXtoFINN())

# add preprocessing
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
chkpt_preproc_name = join(build_dir, "preproc.onnx")
export_qonnx(ToTensor(), torch.randn(ishape), chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = cleanup_model(pre_model)
pre_model = pre_model.transform(ConvertQONNXtoFINN())
model = model.transform(MergeONNXModels(pre_model))
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# final cleanup
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

# onnx resize node workaround
# dummy = oh.make_tensor_value_info("dummy", onnx.TensorProto.FLOAT, [1])
# for n in model.graph.node:
#     if n.op_type == "Resize":
#         n.input[1] = dummy.name
# model.graph.value_info.append(dummy)

model.save('export_tidy.onnx')

/home/vision/danilowi/serious_mot/finn/deps/qonnx/src/qonnx/transformation/merge_onnx_models.py:70: UserWarning: [MergeONNXModels] opsets for models to merge differ: 14 vs 17, output model will use opset 17
  warnings.warn(
/home/vision/danilowi/serious_mot/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [17]:
# showInNetron(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_raw.onnx"))
showInNetron(rawmodel_filename)
# showInNetron('step_slr_floorplan.onnx')

Stopping http://0.0.0.0:4444
Serving '/home/vision/danilowi/serious_mot/finn/notebooks/experiments/yolov8/quantyolov8_4w4a_indact_raw.onnx' at http://0.0.0.0:4444


In [20]:
showInNetron('/home/vision/danilowi/serious_mot/finn/notebooks/experiments/yolov8/quantyolov8_4w4a_comact_tidy.onnx')

Stopping http://0.0.0.0:4444
Serving '/home/vision/danilowi/serious_mot/finn/notebooks/experiments/yolov8/quantyolov8_4w4a_comact_tidy.onnx' at http://0.0.0.0:4444


In [16]:
import onnxruntime as rt

input_dict = {"global_in": test_input_np.astype(np.float32) / 255.0}
sess = rt.InferenceSession(onnx_model.SerializeToString())
output_list = sess.run(None, input_dict)

2025-01-13 19:56:29.881248291 [W:onnxruntime:, graph.cc:1296 Graph] Initializer model.0.bn.weight appears in graph inputs and will not be treated as constant value/weight. This may prevent some of the graph optimizations, like const folding. Move it out of graph inputs if there is no need to override it, by either re-generating the model with latest exporter/converter or with the tool onnxruntime/tools/python/remove_initializer_from_input.py.
2025-01-13 19:56:29.881275547 [W:onnxruntime:, graph.cc:1296 Graph] Initializer model.0.bn.bias appears in graph inputs and will not be treated as constant value/weight. This may prevent some of the graph optimizations, like const folding. Move it out of graph inputs if there is no need to override it, by either re-generating the model with latest exporter/converter or with the tool onnxruntime/tools/python/remove_initializer_from_input.py.
2025-01-13 19:56:29.881283603 [W:onnxruntime:, graph.cc:1296 Graph] Initializer model.0.bn.running_mean appe

Fail: [ONNXRuntimeError] : 1 : FAIL : Fatal error: onnx.brevitas:Quant(-1) is not a registered function/op

In [5]:
# QONNX VERIFICATION
import numpy as np
# import finn.core.onnx_exec as oxe
from onnx import helper as oh
import onnx
from qonnx.core.onnx_exec import execute_onnx

from qonnx.transformation.infer_shapes import InferShapes

# MODEL_PREFIX = "verif_untrained"
# model_file = join(finn_root, "notebooks", "experiments", "yolov8", "verif_noresizehack" + "_quantyolov8.onnx")
# test_input_np = np.load(join(finn_root, "notebooks", "experiments", "yolov8", "test_input_384x640.npy"))
# input_dict = {"global_in": test_input_np.astype(np.float32)}
input_dict = {"global_in": test_input_np.astype(np.float32) / 255.0}
# output_golden = [np.load(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_output_golden_{}.npy".format(i))) for i in range(3)]
# model_file = join(finn_root, "notebooks", "experiments", "yolov8", "untrained_quantyolov8.onnx")
# qonnx_model = ModelWrapper(exportmodel_filename)
qonnx_model = ModelWrapper(rawmodel_filename)
# qonnx_model = qonnx_model.transform(InferShapes())

output_dict = execute_onnx(qonnx_model, input_dict, return_full_exec_context=True)
# for i, (k, v) in enumerate(output_dict.items()):
#     output_golden = np.load(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_output_golden_{}.npy".format(i)))
#     print(v.shape, output_golden.shape, np.mean(np.abs(v - output_golden)), np.max(np.abs(v - output_golden)))

In [10]:
print(output_dict.keys())

dict_keys(['global_in', 'global_out', 'Quant_0_out0', 'Quant_27_out0', 'Quant_1_out0', 'Quant_28_out0', 'Quant_2_out0', 'Quant_29_out0', 'Quant_3_out0', 'Quant_30_out0', 'Quant_4_out0', 'Quant_31_out0', 'Quant_5_out0', 'Quant_32_out0', 'Quant_6_out0', 'Quant_33_out0', 'Quant_7_out0', 'Quant_34_out0', 'Quant_8_out0', 'Quant_35_out0', 'Quant_9_out0', 'Quant_36_out0', 'Quant_10_out0', 'Quant_37_out0', 'Quant_11_out0', 'Quant_38_out0', 'Quant_12_out0', 'Quant_39_out0', 'Quant_13_out0', 'Quant_40_out0', 'Quant_14_out0', 'Quant_41_out0', 'Quant_15_out0', 'Quant_42_out0', 'Quant_16_out0', 'Quant_43_out0', 'Quant_17_out0', 'Quant_44_out0', 'Quant_18_out0', 'Quant_45_out0', 'Quant_19_out0', 'Quant_46_out0', 'Quant_20_out0', 'Quant_47_out0', 'Quant_21_out0', 'Quant_48_out0', 'Quant_22_out0', 'Quant_49_out0', 'Quant_23_out0', 'Quant_50_out0', 'Quant_24_out0', 'Quant_51_out0', 'Quant_25_out0', 'Quant_52_out0', 'Quant_26_out0', 'Quant_0_param0', 'Quant_0_param1', 'Quant_53_param1', 'Quant_27_param2

In [5]:
print(len(saved_features))

22


In [14]:
# testpoints = [
#     'Mul_1_out0',
#     'Mul_3_out0',
#     'Mul_11_out0',
#     'Mul_13_out0',
#     'Mul_25_out0',
#     'Mul_27_out0',
#     'Mul_39_out0',
#     'Mul_41_out0',
#     'Mul_49_out0',
#     'Mul_53_out0'
# ]

# raw:
# testpoints = [
#     'Quant_63_out0',
#     'Quant_64_out0',
#     'Quant_68_out0',
#     'Quant_69_out0',
#     'Quant_75_out0',
#     'Quant_76_out0',
#     'Quant_82_out0',
#     'Quant_83_out0',
#     'Quant_87_out0',
#     'Quant_89_out0'
# ]

# backbone only raw:
testpoints = [
    'Quant_27_out0',
    'Quant_28_out0',
    'Quant_32_out0',
    'Quant_33_out0',
    'Quant_39_out0',
    'Quant_40_out0',
    'Quant_46_out0',
    'Quant_47_out0',
    'Quant_51_out0',
    'global_out'
]


for i, t in enumerate(testpoints):
    # feature = features[i].detach().numpy()
    feature = saved_features[i].detach().numpy()
    # print('feature:', feature)
    out = output_dict[t]
    absdiff = np.abs(feature - out)
    mask = absdiff > 0
    errindices = np.where(absdiff > 0)
    numwrong = np.sum(mask)
    print(t, feature.shape, np.sum(mask) / np.product(mask.shape) * 100, "% wrong ({})".format(np.sum(mask)), "mean error:", np.mean(absdiff), "max error:", np.max(absdiff), "mean nonzero:", np.sum(absdiff) / np.sum(mask))
    for i_wrong in range(numwrong if numwrong <= 100 else 100):
        arg = [errindices[k][i_wrong] for k in range(4)]
        brev = feature[errindices[0][i_wrong], errindices[1][i_wrong], errindices[2][i_wrong], errindices[3][i_wrong]]
        onx = out[errindices[0][i_wrong], errindices[1][i_wrong], errindices[2][i_wrong], errindices[3][i_wrong]]
        print("\t", "brevitas:", brev, "onnx:", onx, "arg", arg)

print('OUTPUTS:')

# outputs = ["global_out", "Add_55_out0", "Add_56_out0", "Add_57_out0", "Add_58_out0"]
# outputs = ["global_out", "Concat_13_out0", "Concat_15_out0"] #yolov8
outputs = ['global_out', 'global_out_1', 'global_out_2']
for i, k in enumerate(outputs):
    print(k)
    v = output_dict[k]
    np.save(MODEL_PREFIX + '/onnx_output_{}.npy'.format(i), v)
    # gold = np.load('{}/{}_trainout{}.npy'.format(MODEL_PREFIX, MODEL_PREFIX, i))
    gold = output_golden[i].detach().cpu().numpy()
    diffs = np.abs(v - gold)
    # output_golden = np.load(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_output_golden_{}.npy".format(i)))
    print(v.shape, gold.shape, np.mean(diffs), np.max(diffs))
    mask = diffs > 0
    print(i, np.sum(mask) / np.product(mask.shape) * 100, "% wrong", "max error:", np.max(diffs))
# for i, (k, v) in enumerate(output_dict.items()):
#     print(k)
    # if k in outputs:
    #     output_golden = np.load(join(finn_root, "notebooks", "experiments", "yolov8", MODEL_PREFIX + "_output_golden_{}.npy".format(i)))
    #     print(v.shape, output_golden.shape, np.mean(np.abs(v - output_golden)), np.max(np.abs(v - output_golden)))

# for i in range(3):
#     a = np.load(join(finn_root, "notebooks", "experiments", "yolov8", "verif_untrained" + "_output_golden_{}.npy".format(i)))
#     b = np.load(join(finn_root, "notebooks", "experiments", "yolov8", "verif_noresizehack" + "_output_golden_{}.npy".format(i)))
#     print(np.max(np.abs(a - b)))

Quant_27_out0 (1, 16, 96, 160) 0.0004069010416666667 % wrong (1) mean error: 9.57419e-08 max error: 0.02352953 mean nonzero: 0.023529529571533203
	 brevitas: 1.8823531 onnx: 1.9058826 arg [0, 12, 38, 110]
Quant_28_out0 (1, 32, 48, 80) 0.0 % wrong (0) mean error: 0.0 max error: 0.0 mean nonzero: nan
Quant_32_out0 (1, 32, 48, 80) 0.0 % wrong (0) mean error: 0.0 max error: 0.0 mean nonzero: nan
Quant_33_out0 (1, 64, 24, 40) 0.0 % wrong (0) mean error: 0.0 max error: 0.0 mean nonzero: nan
Quant_39_out0 (1, 64, 24, 40) 0.0 % wrong (0) mean error: 0.0 max error: 0.0 mean nonzero: nan
Quant_40_out0 (1, 128, 12, 20) 0.0032552083333333335 % wrong (1) mean error: 1.3020836e-05 max error: 0.4000001 mean nonzero: 0.40000009536743164
	 brevitas: 1.2 onnx: 1.6000001 arg [0, 109, 3, 2]
Quant_46_out0 (1, 128, 12, 20) 1.4127604166666667 % wrong (434) mean error: 0.0056510423 max error: 0.4000001 mean nonzero: 0.4000000492219002
	 brevitas: 2.0000002 onnx: 1.6000001 arg [0, 0, 1, 1]
	 brevitas: 3.200000

/tmp/ipykernel_209079/4226976414.py:52: RuntimeWarning: invalid value encountered in divide
  print(t, feature.shape, np.sum(mask) / np.product(mask.shape) * 100, "% wrong ({})".format(np.sum(mask)), "mean error:", np.mean(absdiff), "max error:", np.max(absdiff), "mean nonzero:", np.sum(absdiff) / np.sum(mask))


KeyError: 'global_out_1'

In [ ]:
# INDACT, comact weights, backbone
# Quant_27_out0 0.002034505208333333 % wrong mean error: 7.978451e-08 max error: 0.0039215684
# Quant_28_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_32_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_33_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_39_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_40_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_46_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_47_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_51_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# global_out 0.0 % wrong mean error: 0.0 max error: 0.0
# OUTPUTS:
# global_out
# (1, 256, 6, 10) (256, 6, 10) 0.0 0.0
# 0 0.0 % wrong max error: 0.0
# global_out_1


#INDACT backbone
# Quant_27_out0 0.002034505208333333 % wrong mean error: 7.978451e-08 max error: 0.0039215684
# Quant_28_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_32_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_33_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_39_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_40_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_46_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_47_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_51_out0 0.12369791666666666 % wrong mean error: 8.2465274e-05 max error: 0.06666669
# global_out 0.0 % wrong mean error: 0.0 max error: 0.0
# OUTPUTS:
# global_out
# (1, 256, 6, 10) (256, 6, 10) 0.0 0.0
# 0 0.0 % wrong max error: 0.0


# INDACT, wagi COMACT
# Quant_63_out0 0.0004069010416666667 % wrong mean error: 9.57419e-08 max error: 0.02352953
# Quant_64_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_68_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_69_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_75_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_76_out0 0.0032552083333333335 % wrong mean error: 1.3020836e-05 max error: 0.4000001
# Quant_82_out0 1.4127604166666667 % wrong mean error: 0.0056510423 max error: 0.4000001
# Quant_83_out0 2.0963541666666665 % wrong mean error: 0.0084375 max error: 0.8000001
# Quant_87_out0 7.01171875 % wrong mean error: 0.029921878 max error: 1.6000001
# Quant_89_out0 7.337239583333334 % wrong mean error: 0.03104167 max error: 1.2000002
# OUTPUTS:
# global_out
# (1, 144, 24, 40) (1, 144, 24, 40) 0.28675196 7.379461
# 0 99.29759837962962 % wrong max error: 7.379461
# global_out_1
# (1, 144, 12, 20) (1, 144, 12, 20) 0.33099714 8.620743
# 1 99.97395833333333 % wrong max error: 8.620743
# global_out_2
# (1, 144, 6, 10) (1, 144, 6, 10) 0.2809402 3.7755516
# 2 98.5300925925926 % wrong max error: 3.7755516

##INDACT
# Quant_63_out0 0.002034505208333333 % wrong mean error: 4.78709e-07 max error: 0.02352953
# Quant_64_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_68_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_69_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_75_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_76_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_82_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_83_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_87_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# Quant_89_out0 0.0 % wrong mean error: 0.0 max error: 0.0
# OUTPUTS:
# global_out
# (1, 144, 24, 40) (1, 144, 24, 40) 9.18353e-06 0.03281021
# 0 76.44314236111111 % wrong max error: 0.03281021
# global_out_1
# (1, 144, 12, 20) (1, 144, 12, 20) 7.232296e-06 0.0169487
# 1 5.295138888888888 % wrong max error: 0.0169487
# global_out_2
# (1, 144, 6, 10) (1, 144, 6, 10) 4.072984e-08 7.6293945e-06
# 2 2.175925925925926 % wrong max error: 7.6293945e-06

In [7]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg

from qonnx.transformation.base import Transformation
from finn.transformation.streamline.reorder import (
    MoveLinearPastEltwiseAdd,
    MoveLinearPastFork,
)


# def step_streamline_nonlinear(model: ModelWrapper, cfg: build.DataflowBuildConfig):
#     streamline_transformations = [
#         MoveLinearPastEltwiseAdd(),
#         MoveLinearPastFork(),
#     ]
#     for trn in streamline_transformations:
#         print('hello')
#         model = model.transform(trn)
#         model = model.transform(GiveUniqueNodeNames())
#     return model



model_file = join(finn_root, "notebooks", "experiments", "yolov8", "quantyolov8.onnx")
output_dir = join(build_dir, "output_dir")
intermediate_models = join(output_dir, "intermediate_models")
# folding_config_file = join(finn_root, "notebooks", "experiments", "tfc_2mvau", "folding_intrtl_extrtl.json")
# specialize_layers_config_file = join(finn_root, "notebooks", "experiments", "tfc_2mvau", "specialize_rtl_rtl.json")

build_dataflow_steps = [
    "step_qonnx_to_finn",
    "step_tidy_up",
    "step_streamline",
    # step_streamline_nonlinear,
    # MoveLinearOpsPastSplit,
    # "step_convert_to_hw",
    # "step_create_dataflow_partition",
    # "step_specialize_layers",
    # "step_target_fps_parallelization",
    # "step_apply_folding_config",
    # "step_minimize_bit_width",
    # "step_generate_estimate_reports",
    # "step_hw_codegen",
    # "step_hw_ipgen",
    # "step_set_fifo_depths",
    # "step_create_stitched_ip",
    # "step_measure_rtlsim_performance",
    # "step_out_of_context_synthesis",
    # "step_synthesize_bitfile",
    # "step_make_pynq_driver",
    # "step_deployment_package",
]

#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
cfg = build.DataflowBuildConfig(
    output_dir=output_dir,
    verbose=True,
    standalone_thresholds=True,
    # folding_config_file=folding_config_file,
    # specialize_layers_config_file=specialize_layers_config_file,
    synth_clk_period_ns=10,
    board="ZCU104",
    shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.BITFILE,
        build_cfg.DataflowOutputType.PYNQ_DRIVER,
        build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
    ],
    steps = build_dataflow_steps
)
build.build_dataflow_cfg(model_file, cfg)

Previous run results deleted!
Building dataflow accelerator from /scratch/users/mdaniowi/finn/notebooks/experiments/yolov8/quantyolov8.onnx
Intermediate outputs will be generated in /scratch/users/mdaniowi/build_dirs/yolov8
Final outputs will be generated in /scratch/users/mdaniowi/build_dirs/yolov8/output_dir
Build log is at /scratch/users/mdaniowi/build_dirs/yolov8/output_dir/build_dataflow.log
Running step: step_qonnx_to_finn [1/3]
Running step: step_tidy_up [2/3]
Running step: step_streamline [3/3]
Completed successfully


/scratch/users/mdaniowi/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


0

In [8]:
# STREAMLINE

from copy import deepcopy

import numpy as np

from qonnx.transformation.base import Transformation
from qonnx.transformation.general import(
    SortGraph,
    RemoveUnusedTensors,
)
from qonnx.util.basic import get_by_name
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.general import (
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
)

from finn.transformation.streamline import Streamline
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import (
    MoveLinearPastEltwiseAdd,
    MoveLinearPastFork,
    MoveTransposePastFork,
    MoveTransposePastJoinAdd,
    MoveMulPastJoinAdd,
    MoveOpPastFork,
    MoveScalarMulPastConv,
    # MoveIdenticalOpPastJoinOp,

    MoveScalarLinearPastSplit,
    MoveTransposePastSplit,
    MoveMulPastJoinConcat,
    MoveAddPastJoinConcat,
    MoveTransposePastJoinConcat,

    MoveMulPastMaxPool,
    MakeMaxPoolNHWC,
    MakeScaleResizeNHWC
)

# # model_file = join(finn_root, "notebooks", "experiments", "yolov8", "untrained_quantyolov8.onnx")
# # output_dir = join(build_dir, "output_dir")
# # intermediate_models = join(output_dir, "intermediate_models")

# # model = ModelWrapper(join(intermediate_models, "step_tidy_up.onnx"))
# model = ModelWrapper(model_file)
# model = model.transform(Streamline())
# model = model.transform(MoveAddPastJoinConcat())
# model = model.transform(InferDataLayouts())
# model = model.transform(RemoveUnusedTensors())
# model.save(join(build_dir, "test.onnx"))

# # ---------------- shuffle affine ops 
# # forks:
# model = model.transform(MoveScalarLinearPastSplit())
# #
# model = model.transform(MoveLinearPastFork())
# model = model.transform(InferDataTypes())
# model = model.transform(InferDataLayouts())
# model = model.transform(GiveUniqueNodeNames())
# model.save(join(build_dir, "test1.onnx"))
# # joins:

# # model = model.transform(MoveLinearPastEltwiseAdd(), cleanup=True)
# model = model.transform(MoveMulPastJoinAdd())
# model = model.transform(InferDataTypes())
# model = model.transform(GiveUniqueNodeNames())
# model = model.transform(GiveReadableTensorNames())
# model.save(join(build_dir, "test2.onnx"))
# #
# model = model.transform(MoveMulPastJoinConcat())
# model.save(join(build_dir, "test3.onnx"))
# #
# model = model.transform(Streamline())
# # deal with SPPF
# model = model.transform(MoveLinearPastFork())
# model = model.transform(MoveMulPastMaxPool())
# model = model.transform(MoveLinearPastFork())
# model = model.transform(MoveMulPastMaxPool())
# model = model.transform(MoveMulPastJoinConcat())
# model = model.transform(Streamline())
# model.save(join(build_dir, "test4.onnx"))


# # ------------------ shuffle Transpose
# # forks:
# model = model.transform(LowerConvsToMatMul())
# model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
# model = model.transform(absorb.AbsorbConsecutiveTransposes())
# model.save(join(build_dir, "test5.onnx"))
# model = model.transform(MakeScaleResizeNHWC())
# model.save(join(build_dir, "test5_5.onnx"))
# #
# model = model.transform(MoveTransposePastSplit())
# #
# model = model.transform(MoveTransposePastFork())
# model = model.transform(InferDataLayouts())
# model = model.transform(InferDataTypes())
# model.save(join(build_dir, "test6.onnx"))
# # joins:
# model = model.transform(MoveTransposePastJoinAdd())
# model = model.transform(InferDataLayouts())
# model = model.transform(InferDataTypes())
# model.save(join(build_dir, "test7.onnx"))
# #
# model = model.transform(MoveTransposePastJoinConcat())
# model.save(join(build_dir, "test8.onnx"))
# #
# model = model.transform(absorb.AbsorbConsecutiveTransposes())
# model = model.transform(InferDataTypes())
# model = model.transform(GiveUniqueNodeNames())
# model = model.transform(GiveReadableTensorNames())
# model.save(join(build_dir, "test9.onnx"))
# #
# # SPPF
# model = model.transform(MakeMaxPoolNHWC())
# model = model.transform(MoveTransposePastJoinConcat())
# model = model.transform(absorb.AbsorbConsecutiveTransposes())


def step_yolov8_streamline(model: ModelWrapper):
    model = model.transform(Streamline())
    model = model.transform(MoveAddPastJoinConcat())
    additional_streamline_transformations = [
        # Affine ops
        MoveScalarLinearPastSplit(),
        MoveLinearPastFork(),
        MoveMulPastJoinAdd(),
        MoveMulPastJoinConcat(),
        Streamline(),
        # Affine ops in SPPF
        MoveLinearPastFork(),
        MoveMulPastMaxPool(),
        MoveLinearPastFork(),
        MoveMulPastMaxPool(),
        MoveMulPastJoinConcat(),
        Streamline(),
        # Transposes
        LowerConvsToMatMul(),
        absorb.AbsorbTransposeIntoMultiThreshold(),
        absorb.AbsorbConsecutiveTransposes(),
        MakeScaleResizeNHWC(),
        MoveTransposePastSplit(),
        MoveTransposePastFork(),
        MoveTransposePastJoinAdd(),
        MoveTransposePastJoinConcat(),
        absorb.AbsorbConsecutiveTransposes(),
        # Transposes in SPPF
        MakeMaxPoolNHWC(),
        MoveTransposePastJoinConcat(),
        absorb.AbsorbConsecutiveTransposes()
    ]
    for trn in additional_streamline_transformations:
        model = model.transform(trn)
        model = model.transform(GiveUniqueNodeNames())
        model = model.transform(GiveReadableTensorNames())
        model = model.transform(InferDataTypes())
        model = model.transform(InferDataLayouts())
    return model

model = ModelWrapper(model_file)
model = step_yolov8_streamline(model)
model.save(join(build_dir, MODEL_PREFIX + "_streamlined.onnx"))

In [5]:
# STREAMLINE VERIFICATION
import numpy as np
import finn.core.onnx_exec as oxe
from onnx import helper as oh
import onnx

test_input_np = np.load(join(finn_root, "notebooks", "experiments", "yolov8", "test_input_384x640.npy"))
input_dict = {"global_in": test_input_np.astype(np.float32)}
output_golden = [np.load(join(finn_root, "notebooks", "experiments", "yolov8", "untrained_output_golden_{}.npy".format(i))) for i in range(3)]
model_file = join(build_dir, "tescik.onnx")
qonnx_model = ModelWrapper(model_file)

output_dict = oxe.execute_onnx(qonnx_model, input_dict, return_full_exec_context=False)

In [7]:
for i, (k, v) in enumerate(output_dict.items()):
    print(v.shape, output_golden[i].shape, np.mean(np.abs(v - output_golden[i])))
    # print("output:")
    # print(v)
    # print('golden:')
    # print(output_golden[i])



(1, 144, 48, 80) (1, 144, 48, 80) 7.531831e-07
(1, 144, 24, 40) (1, 144, 24, 40) 1.4861904e-07
(1, 144, 12, 20) (1, 144, 12, 20) 3.451389e-07


In [6]:
showInNetron(join(finn_root, "notebooks", "experiments", "yolov8", "yolov8_output_dir", "intermediate_models", "step_set_fifo_depths.onnx"))

Stopping http://0.0.0.0:4444
Serving '/home/vision/danilowi/serious_mot/finn/notebooks/experiments/yolov8/yolov8_output_dir/intermediate_models/step_set_fifo_depths.onnx' at http://0.0.0.0:4444


In [9]:
showInNetron(join(build_dir, MODEL_PREFIX + "_streamlined.onnx"), "xirengcto03")

Stopping http://0.0.0.0:2222
Serving '/scratch/users/mdaniowi/build_dirs/yolov8/uptoc2f_streamlined.onnx' at http://0.0.0.0:2222


In [3]:
showInNetron(join(build_dir, "test.onnx"), "xirengcto03")

Serving '/scratch/users/mdaniowi/build_dirs/yolov8/test.onnx' at http://0.0.0.0:2222


In [7]:
showInNetron(join(build_dir, "test4.onnx"), "xirengcto03")

Stopping http://0.0.0.0:2222
Serving '/scratch/users/mdaniowi/build_dirs/yolov8/test4.onnx' at http://0.0.0.0:2222


In [5]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil




import json
import numpy as np
import os
import shutil
import warnings
from copy import deepcopy
from distutils.dir_util import copy_tree
from functools import partial
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import (
    ApplyConfig,
    GiveReadableTensorNames,
    GiveUniqueNodeNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.util.cleanup import cleanup_model
from qonnx.util.config import extract_model_config_to_json
from shutil import copy

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
import finn.transformation.streamline.absorb as absorb
from finn.analysis.fpgadataflow.dataflow_performance import dataflow_performance
from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
from finn.analysis.fpgadataflow.hls_synth_res_estimation import hls_synth_res_estimation
from finn.analysis.fpgadataflow.op_and_param_counts import (
    aggregate_dict_keys,
    op_and_param_counts,
)
from finn.analysis.fpgadataflow.post_synth_res import post_synth_res
from finn.analysis.fpgadataflow.res_estimation import (
    res_estimation,
    res_estimation_complete,
)
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    DataflowOutputType,
    ShellFlowType,
    VerificationStepType,
)
from finn.core.onnx_exec import execute_onnx
from finn.core.rtlsim_exec import rtlsim_exec
from finn.core.throughput_test import throughput_test_rtlsim
from finn.transformation.fpgadataflow.annotate_cycles import AnnotateCycles
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP
from finn.transformation.fpgadataflow.derive_characteristic import (
    DeriveCharacteristic,
    DeriveFIFOSizes,
)
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from finn.transformation.fpgadataflow.minimize_accumulator_width import (
    MinimizeAccumulatorWidth,
)
from finn.transformation.fpgadataflow.minimize_weight_bit_width import (
    MinimizeWeightBitWidth,
)
from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.prepare_rtlsim import PrepareRTLSim
from finn.transformation.fpgadataflow.replace_verilog_relpaths import (
    ReplaceVerilogRelPaths,
)
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode
from finn.transformation.fpgadataflow.set_fifo_depths import (
    InsertAndSetFIFODepths,
    RemoveShallowFIFOs,
    SplitLargeFIFOs,
)
from finn.transformation.fpgadataflow.set_folding import SetFolding
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.fpgadataflow.synth_ooc import SynthOutOfContext
from finn.transformation.fpgadataflow.vitis_build import VitisBuild
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.qonnx.quant_act_to_multithreshold import (
    default_filter_function_generator,
)
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC


def my_streamline(model: ModelWrapper, cfg: build.DataflowBuildConfig):
    """Run streamlining on given model. Streamlining involves moving floating point
    scale/shift parameters around, collapsing adjacent ones into a single parameter,
    then absorbing the scale/shift into the following `MultiThreshold` node.
    Streamlining requires careful topology design and cannot be applied to all
    topologies.
    """

    model = model.transform(absorb.AbsorbSignBiasIntoMultiThreshold())
    model = model.transform(Streamline())
    need_lowering = len(model.get_nodes_by_op_type("Conv")) > 0
    if need_lowering:
        model = model.transform(LowerConvsToMatMul())
        model = model.transform(MakeMaxPoolNHWC())
        model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
        model = model.transform(MakeMaxPoolNHWC())
        model = model.transform(absorb.AbsorbConsecutiveTransposes())
    # model = model.transform(ConvertBipolarMatMulToXnorPopcount())
    # model = model.transform(Streamline())
    # # absorb final add-mul nodes into TopK
    # model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
    # model = model.transform(InferDataLayouts())
    # model = model.transform(RemoveUnusedTensors())

    if VerificationStepType.STREAMLINED_PYTHON in cfg._resolve_verification_steps():
        verify_step(model, cfg, "streamlined_python", need_parent=False)

    return model


estimates_output_dir = build_dir + "/output_estimates_only"
estimate_only_dataflow_steps = [
    "step_qonnx_to_finn",
    "step_tidy_up",
    # "step_streamline",
    my_streamline,
    "step_convert_to_hw",
    # "step_create_dataflow_partition",
    "step_specialize_layers",
    "step_target_fps_parallelization",
    "step_apply_folding_config",
    "step_minimize_bit_width",
    "step_generate_estimate_reports",
]

#Delete previous run results if exist
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")


cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    mvau_wwidth_max     = 10000,
    target_fps          = 30,
    synth_clk_period_ns = 10.0,
    # fpga_part           = "xc7z020clg400-1",
    board               = "U250",
    steps               = estimate_only_dataflow_steps,
    shell_flow_type=build_cfg.ShellFlowType.VITIS_ALVEO,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)
build.build_dataflow_cfg(model_file, cfg_estimates)

Previous run results deleted!
Building dataflow accelerator from /scratch/users/mdaniowi/finn/notebooks/experiments/yolov8/quantyolov8.onnx
Intermediate outputs will be generated in /scratch/users/mdaniowi/build_dirs/yolov8
Final outputs will be generated in /scratch/users/mdaniowi/build_dirs/yolov8/output_estimates_only
Build log is at /scratch/users/mdaniowi/build_dirs/yolov8/output_estimates_only/build_dataflow.log
Running step: step_qonnx_to_finn [1/9]
Running step: step_tidy_up [2/9]
Running step: my_streamline [3/9]
Running step: step_convert_to_hw [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]


Traceback (most recent call last):
  File "/scratch/users/mdaniowi/finn/src/finn/builder/build_dataflow.py", line 158, in build_dataflow_cfg
    model = transform_step(model, cfg)
  File "/scratch/users/mdaniowi/finn/src/finn/builder/build_dataflow_steps.py", line 421, in step_target_fps_parallelization
    model = model.transform(
  File "/scratch/users/mdaniowi/finn/deps/qonnx/src/qonnx/core/modelwrapper.py", line 140, in transform
    (transformed_model, model_was_changed) = transformation.apply(transformed_model)
  File "/scratch/users/mdaniowi/finn/src/finn/transformation/fpgadataflow/set_folding.py", line 226, in apply
    perf_dict = model.analysis(dataflow_performance)
  File "/scratch/users/mdaniowi/finn/deps/qonnx/src/qonnx/core/modelwrapper.py", line 124, in analysis
    return analysis_fxn(self)
  File "/scratch/users/mdaniowi/finn/src/finn/analysis/fpgadataflow/dataflow_performance.py", line 71, in dataflow_performance
    max_pred_latency = max(pred_latencies)
  File "/sc

> /scratch/users/mdaniowi/finn/src/finn/analysis/fpgadataflow/dataflow_performance.py(70)<lambda>()
     68                 else:
     69                     # find max of any of predecessors
---> 70                     pred_latencies = map(lambda x: latency_at_node_output[x.name], predecessors)
     71                     max_pred_latency = max(pred_latencies)
     72                 latency_at_node_output[node.name] = node_cycles + max_pred_latency



ipdb>  exit


Build failed


-1

In [3]:
showInNetron(join(build_dir, "test5.onnx"), "xirengcto03")

Serving '/scratch/users/mdaniowi/build_dirs/yolov8/test5.onnx' at http://0.0.0.0:2222


In [4]:
showInNetron(join(build_dir, "tescik.onnx"), "xirengcto03")

Serving '/scratch/users/mdaniowi/build_dirs/yolov8/tescik.onnx' at http://0.0.0.0:2222


In [7]:
showInNetron(join(build_dir, "untrained_streamlined.onnx"), "xirengcto03")

Stopping http://0.0.0.0:2222
Serving '/scratch/users/mdaniowi/build_dirs/yolov8/untrained_streamlined.onnx' at http://0.0.0.0:2222


In [12]:
# CONVERT TO HW
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw


model = ModelWrapper(join(build_dir, MODEL_PREFIX + "_streamlined.onnx"))

# standalone_thresholds = True
# if standalone_thresholds:
#     # doing this first causes all threshold layers to be standalone
#     model = model.transform(to_hw.InferThresholdingLayer())
# # needed for bipolar MatMul layers
# model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
# # needed for non-bipolar MatMul layers
# model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
# model = model.transform(to_hw.InferThresholdingLayer())
# # needed for convolutions -- TODO always exec?
# need_conv = len(model.get_nodes_by_op_type("Im2Col")) > 0
# if need_conv:
#     model = model.transform(to_hw.InferConvInpGen())
#     model = model.transform(to_hw.InferStreamingMaxPool())
#     # model = model.transform(RemoveCNVtoFCFlatten())

# #new
# model = model.transform(to_hw.InferAddStreamsLayer())
# model = model.transform(to_hw.InferConcatLayer())
# model = model.transform(to_hw.InferSplitLayer())
# model = model.transform(to_hw.InferUpsample())

# model = model.transform(GiveUniqueNodeNames())
# model = model.transform(InferDataLayouts())

def step_yolov8_convert_to_hw_layers(model: ModelWrapper):

    model = model.transform(to_hw.InferThresholdingLayer())
    model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
    model = model.transform(to_hw.InferConvInpGen())
    model = model.transform(to_hw.InferPool())
    model = model.transform(to_hw.InferAddStreamsLayer())
    model = model.transform(to_hw.InferConcatLayer())
    model = model.transform(to_hw.InferSplitLayer())
    model = model.transform(to_hw.InferUpsample())
    model = model.transform(to_hw.InferDuplicateStreamsLayer()) 
    
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    model = model.transform(InferDataLayouts())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    return model

model = step_yolov8_convert_to_hw_layers(model)
model.save(join(build_dir, MODEL_PREFIX + "_converted_to_hw.onnx"))

In [13]:
showInNetron(join(build_dir, MODEL_PREFIX + "_converted_to_hw.onnx"), "xirengcto03")

Stopping http://0.0.0.0:2222
Serving '/scratch/users/mdaniowi/build_dirs/yolov8/uptoc2f_converted_to_hw.onnx' at http://0.0.0.0:2222


In [13]:
showInNetron(join(build_dir, MODEL_PREFIX + "_converted_to_hw.onnx"), "xirengcto03")

Stopping http://0.0.0.0:2222
Serving '/scratch/users/mdaniowi/build_dirs/yolov8/uptoc2f_converted_to_hw.onnx' at http://0.0.0.0:2222


In [8]:
showInNetron(join(build_dir, "yolov8_output_dir", "intermediate_models", "step_synthesize_bitfile.onnx"))

Stopping http://0.0.0.0:4444
Serving '/home/vision/danilowi/serious_mot/build_dirs/yolov8_build/yolov8_output_dir/intermediate_models/step_synthesize_bitfile.onnx' at http://0.0.0.0:4444


In [17]:
models = join(build_dir, "yolov8_output_dir_old", "intermediate_models")
!ls {models}

dataflow_parent.onnx		      step_minimize_bit_width.onnx
kernel_partitions		      step_out_of_context_synthesis.onnx
step_apply_folding_config.onnx	      step_set_fifo_depths.onnx
step_create_dataflow_partition.onnx   step_slr_floorplan.onnx
step_create_stitched_ip.onnx	      step_specialize_layers.onnx
step_deployment_package.onnx	      step_synthesize_bitfile.onnx
step_generate_estimate_reports.onnx   step_target_fps_parallelization.onnx
step_hw_codegen.onnx		      step_yolov8_convert_to_hw_layers.onnx
step_hw_ipgen.onnx		      step_yolov8_streamline.onnx
step_make_pynq_driver.onnx	      supported_op_partitions
step_measure_rtlsim_performance.onnx


In [9]:
showInNetron("/home/vision/danilowi/serious_mot/build_dirs/yolov8_build/yolov8_output_dir/intermediate_models/kernel_partitions/partition_5.onnx")

Stopping http://0.0.0.0:4444
Serving '/home/vision/danilowi/serious_mot/build_dirs/yolov8_build/yolov8_output_dir/intermediate_models/kernel_partitions/partition_5.onnx' at http://0.0.0.0:4444


In [18]:
showInNetron(join(models, "step_set_fifo_depths.onnx"))

Stopping http://0.0.0.0:4444
Serving '/home/vision/danilowi/serious_mot/build_dirs/yolov8_trained/yolov8_output_dir_old/intermediate_models/step_set_fifo_depths.onnx' at http://0.0.0.0:4444
